# P2 — Feature engineering: from raw columns to clinical features

**Where we are:** P1 showed us the data. Before modelling, we turn raw columns into features that (a) are **available at prediction time**, (b) are **clean**, and (c) **mean the same thing in a 7-month-old and a 4-year-old**.

**Plan for this notebook**
1. Remove leakage and skip-logic columns
2. Set physiologically impossible values to missing
3. WHO growth z-scores (weight-for-age, height-for-age, weight-for-height, MUAC-for-age)
4. Malnutrition categories (underweight, wasting, SAM)
5. Blantyre Coma Score total
6. Age-adjusted vitals (IMCI fast breathing, lowest SpO₂, shock index)
7. "Was the test done?" indicators
8. Leakage screen: single-feature AUROC
9. Save the feature table for P3

⚠️ Synthetic data (CART/`synthpop`) — methods transfer, exact numbers may not.

Run one cell at a time with **Shift+Enter**.

## 1. Load the data
Same as P1. `y` = 1 if the child died in hospital.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

df = pd.read_csv("PSDC_SyntheticTrainingData_Dataset_ODR.tab", sep="\t")
y = (df["inhospital_mortality"] == 1).astype(int)
print(df.shape, "| deaths:", y.sum())

## 2. Remove leakage and skip-logic columns
**The golden rule:** a feature is allowed only if a clinician would have it **at the moment of prediction** (here: admission).

| Column | Why removed |
|---|---|
| `lengthadm` | Length of stay — known only at discharge (an outcome) |
| `lactate2_mmolpl_adm` | Late lactate, up to 8 h after admission → from the future |
| `studyid_adm` | An ID, not a clinical feature (IDs can leak through ordering) |
| `*source_adm`, `nonexclbreastfed_adm`, `momagefirstpreg_adm`, `spo2other_adm` | Skip-logic follow-up questions — mostly missing *by design* (P1 §6) |

**Grey zone — kept but watch:** admission antibiotics (`admitabx_*`) and oxygen status are decided *by clinicians at admission*. They are available in time, but they encode clinician judgement (confounding by indication). Fine for *prediction*, never for *causal* claims.

**What to look for:** 138 → 128 columns.

In [ ]:
LEAKAGE = ["lengthadm", "lactate2_mmolpl_adm"]
IDS = ["studyid_adm"]
SKIP_LOGIC = ["nonexclbreastfed_adm", "spo2other_adm", "vaccmeaslessource_adm",
              "vaccpneumocsource_adm", "vaccdptsource_adm", "momagefirstpreg_adm"]

X = df.drop(columns=LEAKAGE + IDS + SKIP_LOGIC + ["inhospital_mortality"])
print(f"Kept {X.shape[1]} of {df.shape[1]} columns")

## 3. Impossible values → missing
In P1 we saw glucose = 0 and haematocrit up to 82%. These are recording errors, not biology. We set them to **missing** (not delete the child) and let the imputer handle them later.

Limits are deliberately **wide** — we only remove the *impossible*, not the *abnormal*. An SpO₂ of 57% is terrible but real; deleting it would remove the sickest children.

**What to look for:** 11 glucose and 7 haematocrit values removed; vitals untouched.

In [ ]:
LIMITS = {"glucose_mmolpl_adm": (0.5, 40), "hematocrit_gpdl_adm": (5, 65),
          "hr_bpm_adm": (40, 250), "rr_brpm_app_adm": (8, 120), "sysbp_mmhg_adm": (40, 180),
          "temp_c_adm": (32, 43), "spo2site1_pc_oxi_adm": (40, 100), "spo2site2_pc_oxi_adm": (40, 100)}

for col, (lo, hi) in LIMITS.items():
    bad = X[col].notna() & ~X[col].between(lo, hi)
    print(f"{col:24s} {bad.sum():3d} implausible -> NaN")
    X.loc[bad, col] = np.nan

## 4. WHO growth z-scores
**The problem:** 9 kg is normal for a 9-month-old and severe malnutrition for a 4-year-old. Raw weight mixes *age* and *nutrition*. The model would mostly learn "younger children weigh less".

**The fix:** compare each child to the **WHO Child Growth Standards** for the same sex and age. A z-score says *how many standard deviations* the child is from the WHO median:
- 0 = median, −2 = moderately low, −3 = severely low.

**How (LMS method):** WHO publishes, for every sex × age (in days), three numbers — **L** (skewness), **M** (median), **S** (spread):

$$z = \frac{(X/M)^L - 1}{L \cdot S}$$

For weight and MUAC, WHO adds a correction beyond ±3 SD because the tails are stretched.

The tables are in `who_growth/` (from the official WHO `anthro` R package).

In [ ]:
def lms_z(value, L, M, S, restricted=True):
    z = np.where(L == 0, np.log(value / M) / S, ((value / M) ** L - 1) / (L * S))
    if restricted:   # WHO tail adjustment beyond +/-3 SD (weight & MUAC indicators)
        sd = lambda k: M * (1 + L * S * k) ** (1 / L)
        z = np.where(z > 3, 3 + (value - sd(3)) / (sd(3) - sd(2)), z)
        z = np.where(z < -3, -3 + (value - sd(-3)) / (sd(-2) - sd(-3)), z)
    return z

ref = lambda name: pd.read_csv(f"who_growth/{name}.txt", sep="\t")
ref("weianthro").head()   # sex 1 = boy, 2 = girl; age in days

### 4a. Look up each child's L, M, S and compute z-scores
- Age in months × 30.4375 → age in days (WHO tables are daily).
- Weight-for-height uses **length** tables under 24 months and **height** tables from 24 months (WHO convention: babies are measured lying down).

**What to look for:** mean WAZ ≈ −1.3 — this population is, on average, lighter than the WHO standard (expected for sick children in a low-income setting). A few values fall outside the WHO plausibility limits → set to missing.

In [ ]:
sex = X["sex_adm"].map({"Male": 1, "Female": 2})
age_days = (X["agecalc_adm"] * 30.4375).round().astype(int)
under2 = X["agecalc_adm"] < 24

def by_age(table, value):
    t = ref(table).set_index(["sex", "age"])
    lms = t.reindex(pd.MultiIndex.from_arrays([sex, age_days])).to_numpy()[:, :3].astype(float)
    return lms_z(value.to_numpy(float), *lms.T, restricted=(table != "lenanthro"))

X["waz"] = by_age("weianthro", X["weight_kg_adm"])        # weight-for-age
X["haz"] = by_age("lenanthro", X["height_cm_adm"])        # height-for-age (stunting)
X["muacz"] = by_age("acanthro", X["muac_mm_adm"] / 10)   # MUAC-for-age (table in cm)

# weight-for-length (<2 y) / weight-for-height (>=2 y): looked up by body size, not age
wfl = ref("wflanthro").set_index(["sex", "length"])
wfh = ref("wfhanthro").set_index(["sex", "height"])
size = X["height_cm_adm"].round(1)
lms = np.full((len(X), 3), np.nan)
for tab, mask in [(wfl, under2), (wfh, ~under2)]:
    idx = pd.MultiIndex.from_arrays([sex[mask], size[mask]])
    lms[mask.to_numpy()] = tab.reindex(idx).to_numpy()[:, :3].astype(float)
X["whz"] = lms_z(X["weight_kg_adm"].to_numpy(float), *lms.T)   # wasting

# WHO flags: biologically implausible z-scores -> missing
for col, (lo, hi) in {"waz": (-6, 5), "haz": (-6, 6), "whz": (-5, 5), "muacz": (-5, 5)}.items():
    bad = X[col].notna() & ~X[col].between(lo, hi)
    print(f"{col}: {bad.sum()} implausible -> NaN")
    X.loc[bad, col] = np.nan

X[["waz", "haz", "whz", "muacz"]].describe().round(2)

### 4b. Did the z-score help? Raw weight vs weight-for-age
**What to look for:**
- **Left:** raw weight rises steeply with age — deaths (red) are hidden in a sloped cloud.
- **Right:** WAZ is flat across age — age is removed, and the deaths now sit clearly **below** the survivors.
- Single-feature AUROC improves (weight ≈ 0.585 → WAZ ≈ 0.596). Modest, but the feature now means the same thing at every age, which is what matters for a model and for a clinician reading it.

In [ ]:
d = y == 1
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, col, lab in [(ax[0], "weight_kg_adm", "Weight (kg)"), (ax[1], "waz", "Weight-for-age z-score")]:
    a.scatter(X.loc[~d, "agecalc_adm"], X.loc[~d, col], s=4, alpha=.3, label="survived")
    a.scatter(X.loc[d, "agecalc_adm"], X.loc[d, col], s=14, c="red", label="died")
    a.set_xlabel("Age (months)"); a.set_ylabel(lab); a.legend()
ax[1].axhline(-2, ls="--", c="grey"); ax[1].axhline(-3, ls=":", c="grey")
plt.tight_layout(); plt.show()

auc = lambda col: roc_auc_score(y[X[col].notna()], -X[col].dropna())   # lower = worse
print(f"AUROC  weight: {auc('weight_kg_adm'):.3f}   WAZ: {auc('waz'):.3f}")

## 5. Malnutrition categories (WHO definitions)
Clinicians think in categories, and so do guidelines:

| Feature | Definition |
|---|---|
| `underweight` | WAZ < −2 |
| `wasting` | WHZ < −2 **or** MUAC < 125 mm |
| `sam` (severe acute malnutrition) | WHZ < −3 **or** MUAC < 115 mm |

MUAC is used as a backup because it's the bedside tool in Uganda and it doesn't need height.

**What to look for:** SAM roughly doubles mortality (~8% vs ~4%). About 1 in 3 children is wasted — malnutrition is a major co-driver of sepsis death in this setting.

In [ ]:
X["underweight"] = (X["waz"] < -2).astype(int)
X["wasting"] = ((X["whz"] < -2) | (X["muac_mm_adm"] < 125)).astype(int)
X["sam"] = ((X["whz"] < -3) | (X["muac_mm_adm"] < 115)).astype(int)

def rate_table(cols):
    return pd.DataFrame({c: {"n with sign": X[c].sum(),
                             "mortality % if yes": round(y[X[c] == 1].mean() * 100, 1),
                             "mortality % if no": round(y[X[c] == 0].mean() * 100, 1)} for c in cols}).T

rate_table(["underweight", "wasting", "sam"])

## 6. Blantyre Coma Score (BCS)
The paediatric coma scale for pre-verbal children (Glasgow needs speech). Three parts → total **0–5**:

| Part | Scores |
|---|---|
| Eye | watches/follows = 1, fails = 0 |
| Verbal | cries appropriately = 2, moan/abnormal = 1, none = 0 |
| Motor | localises = 2, withdraws = 1, none = 0 |

BCS ≤ 2 = unrousable coma (used, e.g., in the WHO definition of cerebral malaria).

**Why combine?** Three correlated columns → one ordinal feature = fewer predictors (better EPV) and clinically familiar.

**What to look for:** a steep gradient — BCS 0 ≈ 56% mortality, BCS 5 ≈ 3%. The jump is between 2 and 3, which supports the ≤ 2 cut-off. The small dip at 3 vs 4 comes from tiny numbers (34 and 61 children).

In [ ]:
eye = X["bcseye_adm"].map({"Watches or follows": 1, "Fails to watch or follow": 0})
verbal = X["bcsverbal_adm"].map({"Cries appropriately with pain, or, if verbal, speaks": 2,
                                 "Moan or abnormal cry with pain": 1, "No vocal response to pain": 0})
motor = X["bcsmotor_adm"].map({"Localizes painful stimulus": 2,
                               "Withdraws limb from painful stimulus": 1,
                               "No response or inappropriate response": 0})
X["bcs_total"] = eye + verbal + motor
X["bcs_coma"] = (X["bcs_total"] <= 2).astype(int)

pd.DataFrame({"children": X["bcs_total"].value_counts().sort_index(),
              "deaths": y.groupby(X["bcs_total"]).sum(),
              "mortality_%": (y.groupby(X["bcs_total"]).mean() * 100).round(1)})

## 7. Age-adjusted vitals
Normal respiratory and heart rates **fall with age**. An RR of 45 is normal at 8 months and fast at 3 years.

- **`rr_ratio`** = RR ÷ WHO IMCI fast-breathing cut-off (50/min if < 12 months, 40/min if 12–59 months). > 1 = fast breathing *for age*.
- **`spo2_min`** = the lower of the two oximeter readings (the worst value matters most). **`hypoxaemia`** = SpO₂ < 90% (WHO oxygen threshold).
- **`shock_index`** = HR ÷ systolic BP. It rises when the heart compensates for poor circulation. (In children the normal range is age-dependent — treat it as a relative measure.)

**What to look for:** fast breathing and hypoxaemia raise mortality only modestly here (~5% vs 3–4%). Compare with BCS: in this cohort, **neurological status and nutrition** separate deaths better than vital signs.

In [ ]:
rr_cut = np.where(X["agecalc_adm"] < 12, 50, 40)
X["rr_ratio"] = X["rr_brpm_app_adm"] / rr_cut
X["fast_breathing"] = (X["rr_ratio"] >= 1).astype(int)
X["spo2_min"] = X[["spo2site1_pc_oxi_adm", "spo2site2_pc_oxi_adm"]].min(axis=1)
X["hypoxaemia"] = (X["spo2_min"] < 90).astype(int)
X["shock_index"] = X["hr_bpm_adm"] / X["sysbp_mmhg_adm"]

rate_table(["fast_breathing", "hypoxaemia", "bcs_coma"])

## 8. "Was the test done?" indicators
From P1 §6: missing lactate is **informative** — sicker children get tested. We keep that signal as explicit 0/1 columns *before* imputation erases it.

**What to look for:** mortality 4.7% when lactate was measured vs 3.0% when not.

⚠️ This signal depends on **local practice**. In a hospital that measures lactate in everyone, the indicator means nothing. That's a transportability risk (P6).

In [ ]:
X["lactate_measured"] = X["lactate_mmolpl_adm"].notna().astype(int)
X["hct_measured"] = X["hematocrit_gpdl_adm"].notna().astype(int)
rate_table(["lactate_measured", "hct_measured"])

## 9. Leakage screen: single-feature AUROC
A quick safety check: compute the AUROC of **each feature on its own**. (AUROC = the probability that a random death gets a higher score than a random survivor; 0.5 = coin flip, 1.0 = perfect.)

**Rule of thumb:** if any single admission variable gives AUROC > ~0.85–0.9 for mortality, **be suspicious** — it's usually leakage (e.g., a "discharge status" or a post-event lab), not a miracle.

We use `max(AUC, 1 − AUC)` so direction doesn't matter.

**What to look for:** the top features are 0.58–0.66 — BCS, nutrition z-scores and lactate. Nothing suspicious. Realistic but weak individually → we'll need to **combine** them (P3).

In [ ]:
num = X.select_dtypes("number")
aucs = pd.Series({c: roc_auc_score(y[num[c].notna()], num[c].dropna())
                  for c in num if num[c].nunique() > 1})
aucs = aucs.map(lambda a: max(a, 1 - a)).sort_values(ascending=False)
aucs.head(12).round(3)

## 10. Save the feature table
We save the cleaned, engineered table for P3. We have **not** imputed or scaled yet — that must happen **inside cross-validation** (fit on training folds only), or test-set information leaks into training.

In [ ]:
out = X.assign(inhospital_mortality=y)
out.to_csv("p2_features.csv", index=False)
print("Saved p2_features.csv:", out.shape)
new = ["waz", "haz", "whz", "muacz", "underweight", "wasting", "sam", "bcs_total", "bcs_coma",
       "rr_ratio", "fast_breathing", "spo2_min", "hypoxaemia", "shock_index",
       "lactate_measured", "hct_measured"]
print(len(new), "new engineered features:", new)

## ✍️ Try it yourself
1. **Stunting:** create `stunted = haz < -2`. Does it predict death as strongly as wasting? Why might *acute* malnutrition (wasting) matter more than *chronic* malnutrition (stunting) during an acute infection?
2. **Heart rate for age:** make an `hr_ratio` using a tachycardia cut-off of 160/min under 12 months and 140/min above. Compare its AUROC with raw `hr_bpm_adm`.
3. **Leakage test:** add `df["lengthadm"]` back and run section 9. What AUROC does it get? Would a *high* AUROC here prove leakage, or could a leaky variable also have a *low* AUROC?
4. **Coma cut-off:** try `bcs_total <= 3` instead of `<= 2`. How many more children are flagged, and what happens to mortality in the flagged group?

**Next — P3:** choose ~10 predictors (EPV!), build a scikit-learn `Pipeline` (impute → scale → logistic regression), and compare it with a simple bedside score.